In [ ]:
import torch
import os

# ── Verify GPU ──
print("="*40)
print(f"GPU count : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}    : {torch.cuda.get_device_name(i)}")
    print(f"  VRAM  : {props.total_memory / 1e9:.1f} GB")
total = sum([torch.cuda.get_device_properties(i).total_memory 
             for i in range(torch.cuda.device_count())])
print(f"Total VRAM: {total / 1e9:.1f} GB")
print("="*40)

# ── Check data path ──


In [ ]:
DATA = '/kaggle/input/SemEvel 2025-Task7/'
print(f"\nData path exists: {os.path.exists(DATA)}")
print("\nFiles found:")
for root, dirs, files in os.walk(DATA):
    level = root.replace(DATA, '').count(os.sep)
    indent = '  ' * level
    for f in files:
        size = os.path.getsize(
               os.path.join(root, f)) / (1024*1024)
        print(f"{indent}{f:<45} {size:.1f} MB")

In [ ]:
import os

# Search for your data everywhere
print("Searching for your dataset...")
print("\n/kaggle/input/ contents:")
for item in os.listdir('/kaggle/input/'):
    print(f"  {item}")

In [ ]:
import os

# Look inside datasets folder
for root, dirs, files in os.walk('/kaggle/input/datasets'):
    level = root.replace('/kaggle/input/datasets', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size = os.path.getsize(os.path.join(root, f)) / (1024*1024)
        print(f"{indent}  {f:<45} {size:.1f} MB")

In [ ]:
import pandas as pd
import json
import ast
import os

# ── Paths ──
DATA     = '/kaggle/input/datasets/sayyamsatti/samevel-2025-task-7/'
TRAIN    = DATA + 'train_dev_sets/'
TEST     = DATA + 'test_set/'
SCRIPTS  = DATA + 'scripts/'
OUTPUT   = '/kaggle/working/'

print("Paths set ✓")

# ── Load train/dev files ──
print("\nLoading train/dev files...")
train_posts    = pd.read_csv(TRAIN + 'posts.csv')
train_fc       = pd.read_csv(TRAIN + 'fact_checks.csv')
train_pairs    = pd.read_csv(TRAIN + 'pairs.csv')
dev_pairs_cross = pd.read_csv(TRAIN + 'pairs_dev_crosslingual.csv')
dev_pairs_mono  = pd.read_csv(TRAIN + 'pairs_dev_monolingual.csv')

with open(TRAIN + 'tasks.json', 'r') as f:
    train_tasks = json.load(f)
with open(TRAIN + 'crosslingual_reference.json', 'r') as f:
    cross_ref = json.load(f)
with open(TRAIN + 'monolingual_reference.json', 'r') as f:
    mono_ref = json.load(f)

print("Train/dev loaded ✓")

# ── Load test files ──
print("\nLoading test files...")
test_posts = pd.read_csv(TEST + 'posts.csv')
test_fc    = pd.read_csv(TEST + 'fact_checks.csv')
test_pairs_cross = pd.read_csv(TEST + 'pairs_test_crosslingual.csv')
test_pairs_mono  = pd.read_csv(TEST + 'pairs_test_monolingual.csv')

with open(TEST + 'tasks.json', 'r') as f:
    test_tasks = json.load(f)
with open(TEST + 'crosslingual_reference.json', 'r') as f:
    test_cross_ref = json.load(f)
with open(TEST + 'monolingual_reference.json', 'r') as f:
    test_mono_ref = json.load(f)

print("Test files loaded ✓")

# ── Summary ──
print("\n" + "="*45)
print(f"train_posts      : {len(train_posts):>8,} rows")
print(f"train_fc         : {len(train_fc):>8,} rows")
print(f"train_pairs      : {len(train_pairs):>8,} rows")
print(f"dev_pairs_cross  : {len(dev_pairs_cross):>8,} rows")
print(f"dev_pairs_mono   : {len(dev_pairs_mono):>8,} rows")
print(f"test_posts       : {len(test_posts):>8,} rows")
print(f"test_fc          : {len(test_fc):>8,} rows")
print(f"test_pairs_cross : {len(test_pairs_cross):>8,} rows")
print(f"test_pairs_mono  : {len(test_pairs_mono):>8,} rows")
print("="*45)
print("\nAll files loaded ✓")


In [ ]:
import ast
import re

def parse_text_tuple(val):
    """
    Parses the tuple format in posts/fact_checks:
    ('original text', 'english translation', [('lang', confidence)])
    Returns: (original, english, language_code)
    """
    if pd.isna(val):
        return None, None, 'unk'
    try:
        t = ast.literal_eval(str(val))
        original = t[0] if t[0] else None
        english  = t[1] if t[1] else None
        lang     = t[2][0][0] if t[2] else 'unk'
        return original, english, lang
    except:
        return str(val), str(val), 'unk'

def parse_ocr(val):
    """
    Parses OCR field from posts — list of tuples
    Returns combined OCR text in english
    """
    if pd.isna(val): return ''
    try:
        items = ast.literal_eval(str(val))
        texts = []
        for item in items:
            if item[1]: texts.append(item[1])  # english version
            elif item[0]: texts.append(item[0]) # original if no english
        return ' '.join(texts).strip()
    except:
        return ''

# ── Parse train posts ──
print("Parsing train posts...")
train_posts[['text_orig','text_eng','lang']] = train_posts['text'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
train_posts['ocr_eng'] = train_posts['ocr'].apply(parse_ocr)

# Combine text + ocr (as the paper does)
train_posts['post_text_orig'] = (
    train_posts['text_orig'].fillna('') + ' ' +
    train_posts['ocr_eng'].fillna('')
).str.strip()
train_posts['post_text_eng'] = (
    train_posts['text_eng'].fillna('') + ' ' +
    train_posts['ocr_eng'].fillna('')
).str.strip()
print(f"  Train posts parsed ✓ — {len(train_posts):,} rows")

# ── Parse test posts ──
print("Parsing test posts...")
test_posts[['text_orig','text_eng','lang']] = test_posts['text'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
test_posts['ocr_eng'] = test_posts['ocr'].apply(parse_ocr)
test_posts['post_text_orig'] = (
    test_posts['text_orig'].fillna('') + ' ' +
    test_posts['ocr_eng'].fillna('')
).str.strip()
test_posts['post_text_eng'] = (
    test_posts['text_eng'].fillna('') + ' ' +
    test_posts['ocr_eng'].fillna('')
).str.strip()
print(f"  Test posts parsed ✓ — {len(test_posts):,} rows")

# ── Parse train fact_checks ──
print("Parsing train fact_checks...")
train_fc[['claim_orig','claim_eng','lang']] = train_fc['claim'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
train_fc[['title_orig','title_eng','_']] = train_fc['title'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))

# Combine claim + title (as the paper does)
train_fc['fc_text_orig'] = (
    train_fc['claim_orig'].fillna('') + ' ' +
    train_fc['title_orig'].fillna('')
).str.strip()
train_fc['fc_text_eng'] = (
    train_fc['claim_eng'].fillna('') + ' ' +
    train_fc['title_eng'].fillna('')
).str.strip()
print(f"  Train fact_checks parsed ✓ — {len(train_fc):,} rows")

# ── Parse test fact_checks ──
print("Parsing test fact_checks...")
test_fc[['claim_orig','claim_eng','lang']] = test_fc['claim'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
test_fc[['title_orig','title_eng','_']] = test_fc['title'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
test_fc['fc_text_orig'] = (
    test_fc['claim_orig'].fillna('') + ' ' +
    test_fc['title_orig'].fillna('')
).str.strip()
test_fc['fc_text_eng'] = (
    test_fc['claim_eng'].fillna('') + ' ' +
    test_fc['title_eng'].fillna('')
).str.strip()
print(f"  Test fact_checks parsed ✓ — {len(test_fc):,} rows")

# ── Verify with samples ──
print("\n" + "="*50)
print("SAMPLE PARSED POST:")
sample = train_posts[train_posts['text_orig'].notna()].iloc[0]
print(f"  lang      : {sample['lang']}")
print(f"  original  : {str(sample['post_text_orig'])[:80]}...")
print(f"  english   : {str(sample['post_text_eng'])[:80]}...")

print("\nSAMPLE PARSED FACT-CHECK:")
sample_fc = train_fc[train_fc['fc_text_orig'].notna()].iloc[0]
print(f"  lang      : {sample_fc['lang']}")
print(f"  original  : {str(sample_fc['fc_text_orig'])[:80]}...")
print(f"  english   : {str(sample_fc['fc_text_eng'])[:80]}...")

print("\nLanguage distribution in train posts:")
print(train_posts['lang'].value_counts().head(10))
print("\nLanguage distribution in train fact_checks:")
print(train_fc['lang'].value_counts())

In [ ]:
OUTPUT = '/kaggle/working/'

print("Saving parsed files to /kaggle/working/...")

# ── Save parsed posts ──
train_posts.to_csv(OUTPUT + 'train_posts_parsed.csv', index=False)
print(f"  train_posts_parsed.csv    ✓ — {len(train_posts):,} rows")

test_posts.to_csv(OUTPUT + 'test_posts_parsed.csv', index=False)
print(f"  test_posts_parsed.csv     ✓ — {len(test_posts):,} rows")

# ── Save parsed fact_checks ──
train_fc.to_csv(OUTPUT + 'train_fc_parsed.csv', index=False)
print(f"  train_fc_parsed.csv       ✓ — {len(train_fc):,} rows")

test_fc.to_csv(OUTPUT + 'test_fc_parsed.csv', index=False)
print(f"  test_fc_parsed.csv        ✓ — {len(test_fc):,} rows")

# ── Save pairs ──
train_pairs.to_csv(OUTPUT + 'train_pairs.csv', index=False)
print(f"  train_pairs.csv           ✓ — {len(train_pairs):,} rows")

dev_pairs_cross.to_csv(OUTPUT + 'dev_pairs_cross.csv', index=False)
print(f"  dev_pairs_cross.csv       ✓ — {len(dev_pairs_cross):,} rows")

dev_pairs_mono.to_csv(OUTPUT + 'dev_pairs_mono.csv', index=False)
print(f"  dev_pairs_mono.csv        ✓ — {len(dev_pairs_mono):,} rows")

test_pairs_cross.to_csv(OUTPUT + 'test_pairs_cross.csv', index=False)
print(f"  test_pairs_cross.csv      ✓ — {len(test_pairs_cross):,} rows")

test_pairs_mono.to_csv(OUTPUT + 'test_pairs_mono.csv', index=False)
print(f"  test_pairs_mono.csv       ✓ — {len(test_pairs_mono):,} rows")

# ── Save reference jsons ──
import json
import shutil

shutil.copy(TRAIN + 'crosslingual_reference.json',
            OUTPUT + 'dev_cross_reference.json')
shutil.copy(TRAIN + 'monolingual_reference.json',
            OUTPUT + 'dev_mono_reference.json')
shutil.copy(TEST  + 'crosslingual_reference.json',
            OUTPUT + 'test_cross_reference.json')
shutil.copy(TEST  + 'monolingual_reference.json',
            OUTPUT + 'test_mono_reference.json')
print(f"  reference json files      ✓ — 4 files")

# ── Copy scoring scripts ──
shutil.copy(SCRIPTS + 'crosslingual_scoring.py',
            OUTPUT  + 'crosslingual_scoring.py')
shutil.copy(SCRIPTS + 'monolingual_scoring.py',
            OUTPUT  + 'monolingual_scoring.py')
print(f"  scoring scripts           ✓ — 2 files")

# ── Verify all saved ──
print("\n" + "="*45)
print("FILES IN /kaggle/working/:")
for f in sorted(os.listdir(OUTPUT)):
    size = os.path.getsize(OUTPUT + f) / (1024*1024)
    print(f"  {f:<40} {size:.1f} MB")
print("="*45)
print("\nAll saved permanently ✓")
print("Safe to close Kaggle now if needed.")

In [ ]:
# ── Understand training pairs structure ──
print("Training pairs sample:")
print(train_pairs.head(3).to_string())

print(f"\nColumns: {train_pairs.columns.tolist()}")

# ── Check language distribution of training posts ──
# Merge pairs with posts to see language breakdown
pairs_with_lang = train_pairs.merge(
    train_posts[['post_id','lang','post_text_orig',
                 'post_text_eng']],
    on='post_id', how='left'
)
pairs_with_lang = pairs_with_lang.merge(
    train_fc[['fact_check_id','lang','fc_text_orig',
              'fc_text_eng']],
    on='fact_check_id',
    suffixes=('_post','_fc'),
    how='left'
)

print(f"\nTotal training pairs : {len(pairs_with_lang):,}")
print(f"\nPost language distribution in pairs:")
print(pairs_with_lang['lang_post'].value_counts())

print(f"\nCrossLingual pairs (post lang ≠ fc lang):")
cross = pairs_with_lang[
    pairs_with_lang['lang_post'] != pairs_with_lang['lang_fc']
]
print(f"  Count: {len(cross):,}")
print(f"\nMonolingual pairs (post lang = fc lang):")
mono = pairs_with_lang[
    pairs_with_lang['lang_post'] == pairs_with_lang['lang_fc']
]
print(f"  Count: {len(mono):,}")

print("\nSample crosslingual pair:")
sample = cross.iloc[0]
print(f"  post_lang : {sample['lang_post']}")
print(f"  fc_lang   : {sample['lang_fc']}")
print(f"  post_orig : {str(sample['post_text_orig'])[:60]}...")
print(f"  post_eng  : {str(sample['post_text_eng'])[:60]}...")
print(f"  fc_orig   : {str(sample['fc_text_orig'])[:60]}...")
print(f"  fc_eng    : {str(sample['fc_text_eng'])[:60]}...")

In [ ]:
# ── Build complete pairs dataframe ──
pairs_with_lang = train_pairs.merge(
    train_posts[['post_id','lang','post_text_orig','post_text_eng']],
    on='post_id', how='left'
).merge(
    train_fc[['fact_check_id','lang','fc_text_orig','fc_text_eng']],
    on='fact_check_id',
    suffixes=('_post','_fc'),
    how='left'
)

# ── Drop rows with missing text ──
pairs_with_lang = pairs_with_lang.dropna(
    subset=['post_text_eng','fc_text_eng']
).reset_index(drop=True)

print(f"Total pairs after dropping nulls: {len(pairs_with_lang):,}")

# ────────────────────────────────────────
# CROSSLINGUAL TRAINING DATA
# paper: use ONLY english translations
# ────────────────────────────────────────
cross_train = pairs_with_lang[['post_id',
                                'fact_check_id',
                                'post_text_eng',
                                'fc_text_eng',
                                'lang_post',
                                'lang_fc']].copy()
cross_train.columns = ['post_id','fact_check_id',
                       'query','positive',
                       'lang_post','lang_fc']

# Remove empty texts
cross_train = cross_train[
    (cross_train['query'].str.strip().str.len() > 5) &
    (cross_train['positive'].str.strip().str.len() > 5)
].reset_index(drop=True)

print(f"\nCrosslingual training pairs : {len(cross_train):,}")
print("Sample crosslingual pair:")
s = cross_train.iloc[0]
print(f"  query    : {s['query'][:70]}...")
print(f"  positive : {s['positive'][:70]}...")

# ────────────────────────────────────────
# MONOLINGUAL TRAINING DATA
# paper: use original + english translation
# ────────────────────────────────────────

# Original language pairs
mono_orig = pairs_with_lang[['post_id',
                              'fact_check_id',
                              'post_text_orig',
                              'fc_text_orig',
                              'lang_post',
                              'lang_fc']].copy()
mono_orig.columns = ['post_id','fact_check_id',
                     'query','positive',
                     'lang_post','lang_fc']

# English translation pairs (same pairs, english version)
mono_eng = pairs_with_lang[['post_id',
                             'fact_check_id',
                             'post_text_eng',
                             'fc_text_eng',
                             'lang_post',
                             'lang_fc']].copy()
mono_eng.columns = ['post_id','fact_check_id',
                    'query','positive',
                    'lang_post','lang_fc']

# Combine both — doubles the training data for monolingual
mono_train = pd.concat([mono_orig, mono_eng],
                        ignore_index=True)

# Remove empty texts
mono_train = mono_train[
    (mono_train['query'].str.strip().str.len() > 5) &
    (mono_train['positive'].str.strip().str.len() > 5)
].reset_index(drop=True)

print(f"\nMonolingual training pairs  : {len(mono_train):,}")
print("(original + english = 2x data)")
print("Sample monolingual pair:")
s = mono_train.iloc[0]
print(f"  lang_post: {s['lang_post']}")
print(f"  query    : {s['query'][:70]}...")
print(f"  positive : {s['positive'][:70]}...")

# ── Save both training sets ──
cross_train.to_csv(OUTPUT + 'cross_train_data.csv', index=False)
mono_train.to_csv(OUTPUT + 'mono_train_data.csv', index=False)

print("\n" + "="*45)
print(f"cross_train_data.csv saved : {len(cross_train):,} pairs")
print(f"mono_train_data.csv saved  : {len(mono_train):,} pairs")
print("="*45)
print("Training data ready ✓")

In [ ]:
!pip install sentence-transformers faiss-cpu -q
print("Libraries installed ✓")

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# ── This is the exact model the paper used ──
MODEL_NAME = 'intfloat/multilingual-e5-large-instruct'

print(f"Loading {MODEL_NAME}...")
print("First time: downloads ~2GB, takes 3-5 min")
print("After that: loads from Kaggle cache")

model = SentenceTransformer(MODEL_NAME)
model_dim = model.get_sentence_embedding_dimension()

print(f"\nModel loaded ✓")
print(f"Embedding dimension : {model_dim}")
print(f"Parameters          : ~560M")
print(f"Max sequence length : {model.max_seq_length}")

# ── Move to GPU ──
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Device              : {device}")

# ── Quick sanity test ──
test_texts = [
    "flood evacuation emergency",
    "سیلاب سے بچاؤ",
    "évacuation inondations"
]
vecs = model.encode(test_texts, normalize_embeddings=True)
print(f"\nSanity test ✓")
print(f"Encoded {len(test_texts)} texts → shape {vecs.shape}")

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader
import torch

# ── Load crosslingual training data ──
cross_train = pd.read_csv(OUTPUT + 'cross_train_data.csv')
print(f"Crosslingual training pairs: {len(cross_train):,}")

# ── Build InputExamples ──
print("Building training examples...")
train_examples = []
for _, row in cross_train.iterrows():
    query    = str(row['query']).strip()
    positive = str(row['positive']).strip()
    if len(query) > 5 and len(positive) > 5:
        train_examples.append(
            InputExample(texts=[query, positive])
        )
print(f"Training examples built: {len(train_examples):,}")

# ── DataLoader ──
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

# ── Loss ──
train_loss = MultipleNegativesRankingLoss(model)

# ── Training config ──
NUM_EPOCHS   = 1
WARMUP_STEPS = 100
total_steps  = len(train_dataloader) * NUM_EPOCHS

print(f"\nTraining config:")
print(f"  Epochs     : {NUM_EPOCHS}")
print(f"  Batch size : 16")
print(f"  Steps      : {total_steps:,}")
print(f"  Warmup     : {WARMUP_STEPS}")
print(f"\nStarting crosslingual fine-tuning...")

# ── Fine-tune (use_amp=False fixes the error) ──
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    use_amp=False,
    checkpoint_path=OUTPUT + 'cross_model_checkpoint/',
    checkpoint_save_steps=500,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

# ── Save ──
model.save(OUTPUT + 'cross_model_finetuned/')
print(f"\nCrosslingual model saved ✓")
print(f"Location: {OUTPUT}cross_model_finetuned/")

In [ ]:
import torch
import gc

# ── Delete crosslingual model from GPU memory ──
try:
    del model
    print("Crosslingual model removed from GPU ✓")
except:
    print("Already cleared")

torch.cuda.empty_cache()
gc.collect()

# ── Check free memory now ──
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

In [ ]:
import torch, gc
try: del model_mono
except: pass
torch.cuda.empty_cache()
gc.collect()
print("GPU cleared ✓")

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader
import torch, os

# ── Force single GPU to avoid DataParallel OOM ──
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ── Load fresh model on single GPU ──
print("Loading model on single GPU...")
model_mono = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder='/kaggle/working/model_cache/',
    device='cuda:0'
)
print("Model loaded ✓")

# ── Load data ──
mono_train = pd.read_csv(OUTPUT + 'mono_train_data.csv')
print("Pairs: " + str(len(mono_train)))

# ── Build examples ──
train_examples_mono = []
for _, row in mono_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples_mono.append(InputExample(texts=[q, p]))
print("Examples: " + str(len(train_examples_mono)))

# ── DataLoader ──
train_dataloader_mono = DataLoader(
    train_examples_mono,
    shuffle=True,
    batch_size=16
)

# ── Loss ──
train_loss_mono = MultipleNegativesRankingLoss(model_mono)

total_steps = len(train_dataloader_mono)
print("\nSteps: " + str(total_steps))
print("Est. time: ~2 hours")
print("\nStarting monolingual fine-tuning...")

# ── Fine-tune ──
model_mono.fit(
    train_objectives=[(train_dataloader_mono, train_loss_mono)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    checkpoint_path=OUTPUT + 'mono_model_checkpoint/',
    checkpoint_save_steps=300,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

# ── Save ──
model_mono.save(OUTPUT + 'mono_model_finetuned/')
print("Monolingual model saved ✓")

In [ ]:
import torch, gc, os

# ── Full GPU reset ──
try: del model_mono
except: pass
try: del model
except: pass
try: del train_loss_mono
except: pass
try: del train_dataloader_mono
except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print("GPU " + str(i) + ": " + str(round(free/1e9,1)) + " GB free / " + str(round(total/1e9,1)) + " GB total")

In [ ]:
#restarted...................
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch, gc, pandas as pd
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

OUTPUT = '/kaggle/working/'
print("Setup done ✓")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print("GPU " + str(i) + ": " + str(round(free/1e9,1)) + " GB free")

In [ ]:
mono_train = pd.read_csv(OUTPUT + 'mono_train_data.csv')
print("Pairs loaded: " + str(len(mono_train)))

In [ ]:
model_mono = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder='/kaggle/working/model_cache/',
    device='cuda:0'
)
print("Model loaded ✓")

train_examples_mono = []
for _, row in mono_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples_mono.append(InputExample(texts=[q, p]))
print("Examples: " + str(len(train_examples_mono)))

train_dataloader_mono = DataLoader(
    train_examples_mono,
    shuffle=True,
    batch_size=16
)

train_loss_mono = MultipleNegativesRankingLoss(model_mono)

print("Steps: " + str(len(train_dataloader_mono)))
print("Starting monolingual fine-tuning...")

model_mono.fit(
    train_objectives=[(train_dataloader_mono, train_loss_mono)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    checkpoint_path=OUTPUT + 'mono_model_checkpoint/',
    checkpoint_save_steps=300,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

model_mono.save(OUTPUT + 'mono_model_finetuned/')
print("Monolingual model saved ✓")

In [ ]:
import os
import shutil

# ── Check current disk usage ──
total, used, free = shutil.disk_usage('/kaggle/working/')
print("Disk status:")
print("  Total : " + str(round(total/1e9, 1)) + " GB")
print("  Used  : " + str(round(used/1e9, 1)) + " GB")
print("  Free  : " + str(round(free/1e9, 1)) + " GB")

print("\nFiles by size:")
files = []
for f in os.listdir('/kaggle/working/'):
    path = '/kaggle/working/' + f
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1e6
        files.append((size, f))
    elif os.path.isdir(path):
        dir_size = sum(os.path.getsize(os.path.join(dirpath, filename))
                      for dirpath, dirnames, filenames 
                      in os.walk(path)
                      for filename in filenames) / 1e6
        files.append((dir_size, f + '/'))

for size, name in sorted(files, reverse=True):
    print("  " + str(round(size, 1)).rjust(8) + " MB  " + name)

In [ ]:
import shutil, os

# ── Delete checkpoints (not needed) ──
shutil.rmtree('/kaggle/working/mono_model_checkpoint/')
print("mono_model_checkpoint deleted ✓  freed 11.1 GB")

shutil.rmtree('/kaggle/working/cross_model_checkpoint/')
print("cross_model_checkpoint deleted ✓  freed 6.7 GB")

# ── Verify free space ──
total, used, free = shutil.disk_usage('/kaggle/working/')
print("\nDisk status after cleanup:")
print("  Used : " + str(round(used/1e9, 1)) + " GB")
print("  Free : " + str(round(free/1e9, 1)) + " GB")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# ── Token ──
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch, gc, pandas as pd
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

OUTPUT = '/kaggle/working/'

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print("GPU " + str(i) + ": " + str(round(free/1e9,1)) + " GB free")
print("HuggingFace authenticated ✓")
print("Ready ✓")

In [ ]:
mono_train = pd.read_csv(OUTPUT + 'mono_train_data.csv')
print("Pairs: " + str(len(mono_train)))

model_mono = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder=OUTPUT + 'model_cache/',
    device='cuda:0'
)
print("Model loaded ✓")

train_examples_mono = []
for _, row in mono_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples_mono.append(InputExample(texts=[q, p]))
print("Examples: " + str(len(train_examples_mono)))

train_dataloader_mono = DataLoader(
    train_examples_mono,
    shuffle=True,
    batch_size=16
)
train_loss_mono = MultipleNegativesRankingLoss(model_mono)

print("Steps: " + str(len(train_dataloader_mono)))
print("Starting monolingual fine-tuning...")

model_mono.fit(
    train_objectives=[(train_dataloader_mono, train_loss_mono)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

model_mono.save(OUTPUT + 'mono_model_finetuned/')
print("Monolingual model saved ✓")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, pandas as pd
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

OUTPUT = '/kaggle/working/'

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print("GPU " + str(i) + ": " + str(round(free/1e9,1)) + " GB free")
print("Ready ✓")

In [ ]:
mono_train = pd.read_csv(OUTPUT + 'mono_train_data.csv')

train_examples_mono = []
for _, row in mono_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples_mono.append(InputExample(texts=[q, p]))
print("Examples: " + str(len(train_examples_mono)))

# ── Load model ──
model_mono = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder=OUTPUT + 'model_cache/'
)

# ── Enable gradient checkpointing to save memory ──
model_mono[0].auto_model.gradient_checkpointing_enable()
print("Gradient checkpointing enabled ✓")

# ── Smaller batch size ──
train_dataloader_mono = DataLoader(
    train_examples_mono,
    shuffle=True,
    batch_size=8        # reduced to 8
)
train_loss_mono = MultipleNegativesRankingLoss(model_mono)

print("Steps: " + str(len(train_dataloader_mono)))
print("Starting training...")

model_mono.fit(
    train_objectives=[(train_dataloader_mono, train_loss_mono)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

model_mono.save(OUTPUT + 'mono_model_finetuned/')
print("Monolingual model saved ✓")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc, pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print("GPU " + str(i) + ": " + str(round(free/1e9,1)) + " GB free")
print("Ready ✓")

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc

OUTPUT = '/kaggle/working/'

train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
train_posts = pd.read_csv(OUTPUT + 'train_posts_parsed.csv')
test_posts  = pd.read_csv(OUTPUT + 'test_posts_parsed.csv')
dev_pairs_cross  = pd.read_csv(OUTPUT + 'dev_pairs_cross.csv')
dev_pairs_mono   = pd.read_csv(OUTPUT + 'dev_pairs_mono.csv')
test_pairs_cross = pd.read_csv(OUTPUT + 'test_pairs_cross.csv')
test_pairs_mono  = pd.read_csv(OUTPUT + 'test_pairs_mono.csv')

print("train_fc    : " + str(len(train_fc)))
print("test_fc     : " + str(len(test_fc)))
print("train_posts : " + str(len(train_posts)))
print("test_posts  : " + str(len(test_posts)))
print("All loaded ✓")

In [ ]:
from sentence_transformers import SentenceTransformer

# ── Combine all fact-checks ──
all_fc = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc = all_fc.drop_duplicates(subset=['fact_check_id'])
all_fc = all_fc.reset_index(drop=True)
print("Total unique fact-checks: " + str(len(all_fc)))

# ── Save IDs ──
fc_ids = all_fc['fact_check_id'].tolist()
np.save(OUTPUT + 'fc_ids.npy', np.array(fc_ids))
print("FC IDs saved ✓")

# ── Encode with CROSSLINGUAL model ──
print("\nLoading crosslingual model...")
gc.collect()
torch.cuda.empty_cache()

model_cross = SentenceTransformer(OUTPUT + 'cross_model_finetuned/')
model_cross = model_cross.to('cuda')
print("Model loaded ✓")

texts_eng = all_fc['fc_text_eng'].fillna('').tolist()
print("Encoding " + str(len(texts_eng)) + " fact-checks in English...")

cross_embs = model_cross.encode(
    texts_eng,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)
np.save(OUTPUT + 'cross_fc_embeddings.npy', cross_embs)
print("Cross embeddings saved ✓ shape: " + str(cross_embs.shape))

# ── Free memory ──
del model_cross, cross_embs
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared ✓")

# ── Encode with MONOLINGUAL model ──
print("\nLoading monolingual model...")
model_mono = SentenceTransformer(OUTPUT + 'mono_model_finetuned/')
model_mono = model_mono.to('cuda')
print("Model loaded ✓")

texts_orig = all_fc['fc_text_orig'].fillna('').tolist()
print("Encoding " + str(len(texts_orig)) + " fact-checks in original language...")

mono_embs = model_mono.encode(
    texts_orig,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)
np.save(OUTPUT + 'mono_fc_embeddings.npy', mono_embs)
print("Mono embeddings saved ✓ shape: " + str(mono_embs.shape))

del model_mono, mono_embs
gc.collect()
torch.cuda.empty_cache()

print("\n" + "="*40)
print("All encodings complete ✓")
print("cross_fc_embeddings.npy saved")
print("mono_fc_embeddings.npy saved")
print("fc_ids.npy saved")
print("="*40)

In [ ]:
!pip install faiss-cpu -q
import faiss
import numpy as np

OUTPUT = '/kaggle/working/'

# ── Load embeddings ──
print("Loading embeddings...")
cross_embs = np.load(OUTPUT + 'cross_fc_embeddings.npy').astype('float32')
mono_embs  = np.load(OUTPUT + 'mono_fc_embeddings.npy').astype('float32')
fc_ids     = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)

print("Cross embeddings shape: " + str(cross_embs.shape))
print("Mono embeddings shape : " + str(mono_embs.shape))
print("FC IDs count          : " + str(len(fc_ids)))

# ── Build CROSSLINGUAL FAISS index ──
print("\nBuilding crosslingual FAISS index...")
cross_index = faiss.IndexFlatIP(cross_embs.shape[1])
cross_index.add(cross_embs)
print("Vectors added: " + str(cross_index.ntotal))
faiss.write_index(cross_index, OUTPUT + 'cross_faiss.index')
print("cross_faiss.index saved ✓")

del cross_embs, cross_index

# ── Build MONOLINGUAL FAISS index ──
print("\nBuilding monolingual FAISS index...")
mono_index = faiss.IndexFlatIP(mono_embs.shape[1])
mono_index.add(mono_embs)
print("Vectors added: " + str(mono_index.ntotal))
faiss.write_index(mono_index, OUTPUT + 'mono_faiss.index')
print("mono_faiss.index saved ✓")

del mono_embs, mono_index

print("\n" + "="*40)
print("FAISS indexes built ✓")
print("cross_faiss.index saved")
print("mono_faiss.index saved")
print("="*40)

In [ ]:
import json
import torch
import numpy as np
import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer
import gc

OUTPUT = '/kaggle/working/'

# ── Load everything ──
print("Loading models and indexes...")
gc.collect()
torch.cuda.empty_cache()

# Load IDs
fc_ids = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)
fc_ids_list = [int(x) for x in fc_ids]

# Load FAISS indexes
cross_index = faiss.read_index(OUTPUT + 'cross_faiss.index')
mono_index  = faiss.read_index(OUTPUT + 'mono_faiss.index')
print("FAISS indexes loaded ✓")

# Load models
model_cross = SentenceTransformer(OUTPUT + 'cross_model_finetuned/')
model_mono  = SentenceTransformer(OUTPUT + 'mono_model_finetuned/')
print("Models loaded ✓")

# Load posts
dev_posts  = pd.read_csv(OUTPUT + 'train_posts_parsed.csv')
test_posts = pd.read_csv(OUTPUT + 'test_posts_parsed.csv')
print("Posts loaded ✓")

# ── Retrieval functions ──
def retrieve_crosslingual(query_text, top_k=10):
    """Use english translation → search cross index"""
    q_vec = model_cross.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    scores, indices = cross_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in indices[0]]

def retrieve_monolingual(query_text, top_k=10):
    """Use original text → search mono index"""
    q_vec = model_mono.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    scores, indices = mono_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in indices[0]]

print("\nRetrieval functions ready ✓")

# ── Quick test ──
print("\nTesting retrieval...")
test_query_eng  = "floods in Germany killing babies"
test_query_orig = "Las inundaciones en Alemania mataron bebés"

cross_results = retrieve_crosslingual(test_query_eng)
mono_results  = retrieve_monolingual(test_query_orig)

print("Crosslingual top 5: " + str(cross_results[:5]))
print("Monolingual top 5 : " + str(mono_results[:5]))
print("\nRetrieval working ✓")


In [ ]:
print("Total fc_ids  : " + str(len(fc_ids_list)))
print("Unique fc_ids : " + str(len(set(fc_ids_list))))

# Check if 372897 appears multiple times
count = fc_ids_list.count(372897)
print("372897 appears : " + str(count) + " times")

In [ ]:
# Test with specific queries
test_cases = [
    ("cross", "600 dead babies found after floods in Germany"),
    ("cross", "COVID vaccine causes infertility"),
    ("cross", "earthquake killed hundreds in Turkey"),
    ("mono",  "Las inundaciones en Alemania mataron bebés"),
    ("mono",  "La vacuna COVID causa infertilidad"),
]

for mode, query in test_cases:
    if mode == "cross":
        results = retrieve_crosslingual(query, top_k=5)
    else:
        results = retrieve_monolingual(query, top_k=5)
    
    # Check if results are diverse
    unique = len(set(results))
    print("\n[" + mode + "] " + query[:50])
    print("  Top 5 IDs : " + str(results))
    print("  Unique IDs: " + str(unique) + "/5")
    

In [ ]:
import numpy as np
import faiss

# ── Check embeddings ──
cross_embs = np.load(OUTPUT + 'cross_fc_embeddings.npy').astype('float32')
print("Shape: " + str(cross_embs.shape))

# Check if all vectors are identical
print("First vector sample : " + str(cross_embs[0][:5]))
print("Second vector sample: " + str(cross_embs[1][:5]))
print("Third vector sample : " + str(cross_embs[2][:5]))

# Check variance
print("\nMean of all vectors : " + str(cross_embs.mean()))
print("Std of all vectors  : " + str(cross_embs.std()))

# Check if vectors are normalized
norms = np.linalg.norm(cross_embs[:10], axis=1)
print("Norms of first 10   : " + str(norms))

# Find what index 372897 maps to
fc_ids = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)
fc_ids_list = [int(x) for x in fc_ids]
idx = fc_ids_list.index(372897)
print("\n372897 is at index  : " + str(idx))
print("Its vector sample   : " + str(cross_embs[idx][:5]))

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
import faiss
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

# ── Reload fact-checks fresh ──
train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')

all_fc = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc = all_fc.drop_duplicates(subset=['fact_check_id'])
all_fc = all_fc.reset_index(drop=True)
print("Total unique fc: " + str(len(all_fc)))

# ── Check text columns ──
print("\nfc_text_eng nulls : " + str(all_fc['fc_text_eng'].isna().sum()))
print("fc_text_orig nulls: " + str(all_fc['fc_text_orig'].isna().sum()))

# ── Fill nulls ──
all_fc['fc_text_eng']  = all_fc['fc_text_eng'].fillna('unknown')
all_fc['fc_text_orig'] = all_fc['fc_text_orig'].fillna('unknown')

# ── Verify texts are valid ──
texts_eng  = all_fc['fc_text_eng'].tolist()
texts_orig = all_fc['fc_text_orig'].tolist()

print("\nSample eng text 0 : " + str(texts_eng[0])[:80])
print("Sample eng text 1 : " + str(texts_eng[1])[:80])
print("Sample orig text 0: " + str(texts_orig[0])[:80])

# ── Verify no NaN strings ──
nan_count = sum(1 for t in texts_eng if str(t) == 'nan')
print("\nNaN strings in eng: " + str(nan_count))

In [ ]:
# ── Test encoding on small batch first ──
print("Testing encoding on 10 samples...")
gc.collect()
torch.cuda.empty_cache()

model_cross = SentenceTransformer(OUTPUT + 'cross_model_finetuned/')
model_cross = model_cross.to('cuda')

# Test on first 10
test_texts = texts_eng[:10]
test_embs = model_cross.encode(
    test_texts,
    batch_size=10,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')

print("Test embeddings shape: " + str(test_embs.shape))
print("Any NaN: " + str(np.isnan(test_embs).any()))
print("Sample vector: " + str(test_embs[0][:5]))
print("Norms: " + str(np.linalg.norm(test_embs[:5], axis=1)))

In [ ]:
# ── Test with base model (not fine-tuned) ──
del model_cross
gc.collect()
torch.cuda.empty_cache()

print("Testing with BASE model...")
model_base = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder=OUTPUT + 'model_cache/'
)
model_base = model_base.to('cuda')

test_embs_base = model_base.encode(
    texts_eng[:10],
    batch_size=10,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')

print("Any NaN: " + str(np.isnan(test_embs_base).any()))
print("Sample vector: " + str(test_embs_base[0][:5]))
print("Norms: " + str(np.linalg.norm(test_embs_base[:5], axis=1)))

In [ ]:
import os

# ── Check fine-tuned model files ──
model_path = OUTPUT + 'cross_model_finetuned/'
print("Files in cross_model_finetuned/:")
for f in os.listdir(model_path):
    size = os.path.getsize(model_path + f) / (1024*1024)
    print("  " + f + " — " + str(round(size, 1)) + " MB")

In [ ]:
# ── Check mono model ──
model_path = OUTPUT + 'mono_model_finetuned/'
print("Files in mono_model_finetuned/:")
for f in os.listdir(model_path):
    size = os.path.getsize(model_path + f) / (1024*1024)
    print("  " + f + " — " + str(round(size, 1)) + " MB")

# ── Test mono model too ──
del model_base
gc.collect()
torch.cuda.empty_cache()

from sentence_transformers import SentenceTransformer
model_mono_test = SentenceTransformer(OUTPUT + 'mono_model_finetuned/')
model_mono_test = model_mono_test.to('cuda')

test_embs_mono = model_mono_test.encode(
    texts_orig[:10],
    batch_size=10,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')

print("\nMono model test:")
print("Any NaN: " + str(np.isnan(test_embs_mono).any()))
print("Sample : " + str(test_embs_mono[0][:5]))

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc, pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

OUTPUT = '/kaggle/working/'

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print("GPU " + str(i) + ": " + str(round(free/1e9,1)) + " GB free")
print("Ready ✓")

In [ ]:
# ── Load data ──
cross_train = pd.read_csv(OUTPUT + 'cross_train_data.csv')
print("Pairs: " + str(len(cross_train)))

# ── Build examples ──
train_examples = []
for _, row in cross_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples.append(InputExample(texts=[q, p]))
print("Examples: " + str(len(train_examples)))

# ── Load base model ──
model = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder=OUTPUT + 'model_cache/'
)

# ── Enable gradient checkpointing ──
model[0].auto_model.gradient_checkpointing_enable()
print("Gradient checkpointing enabled ✓")

# ── DataLoader ──
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=8
)
train_loss = MultipleNegativesRankingLoss(model)

print("Steps: " + str(len(train_dataloader)))
print("Starting crosslingual training...")

# ── Train ──
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

# ── Verify before saving ──
print("\nVerifying model before saving...")
test_emb = model.encode(
    ["flood evacuation"],
    normalize_embeddings=True,
    convert_to_numpy=True
)
if np.isnan(test_emb).any():
    print("ERROR: Model produced NaN — DO NOT SAVE")
else:
    print("Model verified ✓ no NaN")
    model.save(OUTPUT + 'cross_model_finetuned/')
    print("Crosslingual model saved ✓")

In [ ]:
import os, gc, torch, numpy as np, faiss, pandas as pd
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

gc.collect()
torch.cuda.empty_cache()

# ── Load base model ──
print("Loading base model...")
model = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder=OUTPUT + 'model_cache/'
)
model = model.to('cuda')
print("Model loaded ✓")

# ── Verify no NaN ──
test = model.encode(
    ["flood evacuation emergency"],
    normalize_embeddings=True,
    convert_to_numpy=True
)
print("NaN check: " + str(np.isnan(test).any()))
print("Sample   : " + str(test[0][:5]))

# ── Load fact-checks ──
train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
all_fc   = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc   = all_fc.drop_duplicates(subset=['fact_check_id']).reset_index(drop=True)
all_fc['fc_text_eng']  = all_fc['fc_text_eng'].fillna('unknown')
all_fc['fc_text_orig'] = all_fc['fc_text_orig'].fillna('unknown')
print("Fact-checks: " + str(len(all_fc)))

# ── Encode for CROSSLINGUAL (english) ──
print("\nEncoding for crosslingual (english)...")
cross_embs = model.encode(
    all_fc['fc_text_eng'].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print("NaN in cross_embs: " + str(np.isnan(cross_embs).any()))
np.save(OUTPUT + 'cross_fc_embeddings.npy', cross_embs)
print("Cross embeddings saved ✓ shape: " + str(cross_embs.shape))
del cross_embs
gc.collect()

# ── Encode for MONOLINGUAL (original) ──
print("\nEncoding for monolingual (original)...")
mono_embs = model.encode(
    all_fc['fc_text_orig'].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print("NaN in mono_embs: " + str(np.isnan(mono_embs).any()))
np.save(OUTPUT + 'mono_fc_embeddings.npy', mono_embs)
print("Mono embeddings saved ✓ shape: " + str(mono_embs.shape))

fc_ids = all_fc['fact_check_id'].tolist()
np.save(OUTPUT + 'fc_ids.npy', np.array(fc_ids))
print("FC IDs saved ✓")

del mono_embs
gc.collect()
torch.cuda.empty_cache()

# ── Build FAISS indexes ──
print("\nBuilding FAISS indexes...")
cross_embs = np.load(OUTPUT + 'cross_fc_embeddings.npy').astype('float32')
cross_index = faiss.IndexFlatIP(cross_embs.shape[1])
cross_index.add(cross_embs)
faiss.write_index(cross_index, OUTPUT + 'cross_faiss.index')
print("Cross FAISS: " + str(cross_index.ntotal) + " vectors ✓")
del cross_embs, cross_index

mono_embs = np.load(OUTPUT + 'mono_fc_embeddings.npy').astype('float32')
mono_index = faiss.IndexFlatIP(mono_embs.shape[1])
mono_index.add(mono_embs)
faiss.write_index(mono_index, OUTPUT + 'mono_faiss.index')
print("Mono FAISS : " + str(mono_index.ntotal) + " vectors ✓")
del mono_embs, mono_index

print("\n" + "="*40)
print("All done ✓")
print("Base model used for both tasks")
print("="*40)

In [ ]:
!pip install faiss-cpu sentence-transformers -q

import os, gc, torch, numpy as np, faiss, pandas as pd
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

gc.collect()
torch.cuda.empty_cache()

# ── Load base model ──
print("Loading base model...")
model = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder=OUTPUT + 'model_cache/'
)
model = model.to('cuda')
print("Model loaded ✓")

# ── Verify no NaN ──
test = model.encode(
    ["flood evacuation emergency"],
    normalize_embeddings=True,
    convert_to_numpy=True
)
print("NaN check: " + str(np.isnan(test).any()))
print("Sample   : " + str(test[0][:5]))

# ── Load fact-checks ──
train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
all_fc   = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc   = all_fc.drop_duplicates(subset=['fact_check_id']).reset_index(drop=True)
all_fc['fc_text_eng']  = all_fc['fc_text_eng'].fillna('unknown')
all_fc['fc_text_orig'] = all_fc['fc_text_orig'].fillna('unknown')
print("Fact-checks: " + str(len(all_fc)))

# ── Encode crosslingual (english) ──
print("\nEncoding for crosslingual (english)...")
cross_embs = model.encode(
    all_fc['fc_text_eng'].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print("NaN in cross_embs: " + str(np.isnan(cross_embs).any()))
np.save(OUTPUT + 'cross_fc_embeddings.npy', cross_embs)
print("Cross embeddings saved ✓ shape: " + str(cross_embs.shape))
del cross_embs
gc.collect()

# ── Encode monolingual (original) ──
print("\nEncoding for monolingual (original)...")
mono_embs = model.encode(
    all_fc['fc_text_orig'].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print("NaN in mono_embs: " + str(np.isnan(mono_embs).any()))
np.save(OUTPUT + 'mono_fc_embeddings.npy', mono_embs)
print("Mono embeddings saved ✓ shape: " + str(mono_embs.shape))

fc_ids = all_fc['fact_check_id'].tolist()
np.save(OUTPUT + 'fc_ids.npy', np.array(fc_ids))
print("FC IDs saved ✓")

del mono_embs
gc.collect()
torch.cuda.empty_cache()

# ── Build FAISS indexes ──
print("\nBuilding FAISS indexes...")
cross_embs = np.load(OUTPUT + 'cross_fc_embeddings.npy').astype('float32')
cross_index = faiss.IndexFlatIP(cross_embs.shape[1])
cross_index.add(cross_embs)
faiss.write_index(cross_index, OUTPUT + 'cross_faiss.index')
print("Cross FAISS: " + str(cross_index.ntotal) + " vectors ✓")
del cross_embs, cross_index

mono_embs = np.load(OUTPUT + 'mono_fc_embeddings.npy').astype('float32')
mono_index = faiss.IndexFlatIP(mono_embs.shape[1])
mono_index.add(mono_embs)
faiss.write_index(mono_index, OUTPUT + 'mono_faiss.index')
print("Mono FAISS : " + str(mono_index.ntotal) + " vectors ✓")
del mono_embs, mono_index

print("\n" + "="*40)
print("All done ✓")
print("Base model used for both tasks")
print("="*40)

In [ ]:
import json
import numpy as np
import faiss
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

# ── Load everything ──
print("Loading...")
model = SentenceTransformer(
    'intfloat/multilingual-e5-large-instruct',
    cache_folder=OUTPUT + 'model_cache/'
)
model = model.to('cuda')

fc_ids      = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)
fc_ids_list = [int(x) for x in fc_ids]

cross_index = faiss.read_index(OUTPUT + 'cross_faiss.index')
mono_index  = faiss.read_index(OUTPUT + 'mono_faiss.index')

test_posts  = pd.read_csv(OUTPUT + 'test_posts_parsed.csv')
train_posts = pd.read_csv(OUTPUT + 'train_posts_parsed.csv')
all_posts   = pd.concat([train_posts, test_posts], ignore_index=True)

print("Model loaded ✓")
print("FAISS cross: " + str(cross_index.ntotal) + " vectors")
print("FAISS mono : " + str(mono_index.ntotal) + " vectors")

# ── Retrieval functions ──
def retrieve_crosslingual(query_eng, top_k=10):
    q_vec = model.encode(
        [query_eng],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    _, indices = cross_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in indices[0]]

def retrieve_monolingual(query_orig, top_k=10):
    q_vec = model.encode(
        [query_orig],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    _, indices = mono_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in indices[0]]

print("\nRetrieval functions ready ✓")

# ── Test ──
r1 = retrieve_crosslingual("floods in Germany killing babies")
r2 = retrieve_monolingual("Las inundaciones en Alemania mataron bebés")
print("Cross top 5: " + str(r1[:5]))
print("Mono top 5 : " + str(r2[:5]))
print("Unique cross: " + str(len(set(r1[:5]))))
print("Unique mono : " + str(len(set(r2[:5]))))

In [ ]:
import json
from tqdm import tqdm

# ── Load dev pairs ──
dev_pairs_cross = pd.read_csv(OUTPUT + 'dev_pairs_cross.csv')
dev_pairs_mono  = pd.read_csv(OUTPUT + 'dev_pairs_mono.csv')

# ── Load reference files ──
with open(OUTPUT + 'dev_cross_reference.json', 'r') as f:
    cross_ref = json.load(f)
with open(OUTPUT + 'dev_mono_reference.json', 'r') as f:
    mono_ref = json.load(f)

print("Dev crosslingual pairs : " + str(len(dev_pairs_cross)))
print("Dev monolingual pairs  : " + str(len(dev_pairs_mono)))

# ── Get dev post IDs ──
dev_cross_post_ids = dev_pairs_cross['post_id'].unique().tolist()
dev_mono_post_ids  = dev_pairs_mono['post_id'].unique().tolist()

print("Dev cross posts: " + str(len(dev_cross_post_ids)))
print("Dev mono posts : " + str(len(dev_mono_post_ids)))

# ── Run crosslingual retrieval on dev ──
print("\nRunning crosslingual retrieval on dev set...")
cross_results = {}
for post_id in tqdm(dev_cross_post_ids):
    post_row = all_posts[all_posts['post_id'] == post_id]
    if post_row.empty:
        continue
    query = str(post_row.iloc[0]['post_text_eng']).strip()
    if len(query) < 3:
        query = str(post_row.iloc[0]['post_text_orig']).strip()
    retrieved = retrieve_crosslingual(query, top_k=10)
    cross_results[str(post_id)] = retrieved

# ── Run monolingual retrieval on dev ──
print("\nRunning monolingual retrieval on dev set...")
mono_results = {}
for post_id in tqdm(dev_mono_post_ids):
    post_row = all_posts[all_posts['post_id'] == post_id]
    if post_row.empty:
        continue
    query = str(post_row.iloc[0]['post_text_orig']).strip()
    if len(query) < 3:
        query = str(post_row.iloc[0]['post_text_eng']).strip()
    retrieved = retrieve_monolingual(query, top_k=10)
    mono_results[str(post_id)] = retrieved

# ── Save results ──
with open(OUTPUT + 'dev_cross_results.json', 'w') as f:
    json.dump(cross_results, f)
with open(OUTPUT + 'dev_mono_results.json', 'w') as f:
    json.dump(mono_results, f)

print("\nResults saved ✓")
print("Cross results: " + str(len(cross_results)) + " posts")
print("Mono results : " + str(len(mono_results)) + " posts")

# ── Evaluate with Success@10 ──
def success_at_k(results, reference, k=10):
    scores = []
    for post_id, retrieved in results.items():
        if post_id not in reference:
            continue
        relevant = set(reference[post_id])
        retrieved_k = set(retrieved[:k])
        hit = 1 if len(relevant & retrieved_k) > 0 else 0
        scores.append(hit)
    return sum(scores) / len(scores) if scores else 0

cross_s10 = success_at_k(cross_results, cross_ref)
mono_s10  = success_at_k(mono_results, mono_ref)

print("\n" + "="*45)
print("DEV SET EVALUATION RESULTS")
print("="*45)
print("Crosslingual S@10 : " + str(round(cross_s10, 4)))
print("Monolingual  S@10 : " + str(round(mono_s10, 4)))
print("="*45)

In [ ]:
# ── Load test posts ──
test_posts_df = pd.read_csv(OUTPUT + 'test_posts_parsed.csv')

with open(OUTPUT + 'test_cross_reference.json', 'r') as f:
    test_cross_ref = json.load(f)
with open(OUTPUT + 'test_mono_reference.json', 'r') as f:
    test_mono_ref = json.load(f)

test_pairs_cross = pd.read_csv(OUTPUT + 'test_pairs_cross.csv')
test_pairs_mono  = pd.read_csv(OUTPUT + 'test_pairs_mono.csv')

test_cross_post_ids = test_pairs_cross['post_id'].unique().tolist()
test_mono_post_ids  = test_pairs_mono['post_id'].unique().tolist()

print("Test cross posts: " + str(len(test_cross_post_ids)))
print("Test mono posts : " + str(len(test_mono_post_ids)))

# ── Run crosslingual on test ──
print("\nRunning crosslingual on test set...")
test_cross_results = {}
for post_id in tqdm(test_cross_post_ids):
    post_row = test_posts_df[test_posts_df['post_id'] == post_id]
    if post_row.empty:
        continue
    query = str(post_row.iloc[0]['post_text_eng']).strip()
    if len(query) < 3:
        query = str(post_row.iloc[0]['post_text_orig']).strip()
    retrieved = retrieve_crosslingual(query, top_k=10)
    test_cross_results[str(post_id)] = retrieved

# ── Run monolingual on test ──
print("\nRunning monolingual on test set...")
test_mono_results = {}
for post_id in tqdm(test_mono_post_ids):
    post_row = test_posts_df[test_posts_df['post_id'] == post_id]
    if post_row.empty:
        continue
    query = str(post_row.iloc[0]['post_text_orig']).strip()
    if len(query) < 3:
        query = str(post_row.iloc[0]['post_text_eng']).strip()
    retrieved = retrieve_monolingual(query, top_k=10)
    test_mono_results[str(post_id)] = retrieved

# ── Save ──
with open(OUTPUT + 'test_cross_results.json', 'w') as f:
    json.dump(test_cross_results, f)
with open(OUTPUT + 'test_mono_results.json', 'w') as f:
    json.dump(test_mono_results, f)

# ── Evaluate ──
test_cross_s10 = success_at_k(test_cross_results, test_cross_ref)
test_mono_s10  = success_at_k(test_mono_results,  test_mono_ref)

print("\n" + "="*45)
print("TEST SET EVALUATION RESULTS")
print("="*45)
print("Crosslingual S@10 : " + str(round(test_cross_s10, 4)))
print("Monolingual  S@10 : " + str(round(test_mono_s10, 4)))
print("="*45)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch, gc, pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

OUTPUT = '/kaggle/working/'

gc.collect()
torch.cuda.empty_cache()

# ── Use smaller stable model ──
# paraphrase-multilingual-mpnet-base-v2
# 278M params, stable training, no NaN issues
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

print("Loading model: " + MODEL_NAME)
model = SentenceTransformer(
    MODEL_NAME,
    cache_folder=OUTPUT + 'model_cache/',
    device='cuda:0'
)
print("Model loaded ✓")
print("Dim: " + str(model.get_sentence_embedding_dimension()))

# ── Quick NaN test first ──
test = model.encode(
    ["flood evacuation emergency"],
    normalize_embeddings=True,
    convert_to_numpy=True
)
print("Base NaN check: " + str(np.isnan(test).any()))
print("Sample: " + str(test[0][:5]))

# ── Load training data ──
cross_train = pd.read_csv(OUTPUT + 'cross_train_data.csv')

train_examples = []
for _, row in cross_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples.append(InputExample(texts=[q, p]))
print("Examples: " + str(len(train_examples)))

# ── DataLoader ──
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=32
)
train_loss = MultipleNegativesRankingLoss(model)

print("Steps: " + str(len(train_dataloader)))
print("Starting training...")

# ── Train ──
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

# ── Verify ──
print("\nVerifying...")
test_emb = model.encode(
    ["flood evacuation emergency"],
    normalize_embeddings=True,
    convert_to_numpy=True
)

if np.isnan(test_emb).any():
    print("ERROR: NaN — not saving")
else:
    model.save(OUTPUT + 'cross_model_mpnet/')
    print("Model saved ✓ — no NaN")
    print("Sample: " + str(test_emb[0][:5]))

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# ── Load monolingual training data ──
mono_train = pd.read_csv(OUTPUT + 'mono_train_data.csv')

train_examples_mono = []
for _, row in mono_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples_mono.append(InputExample(texts=[q, p]))
print("Examples: " + str(len(train_examples_mono)))

# ── Load fresh model ──
del model
gc.collect()
torch.cuda.empty_cache()

model_mono = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    cache_folder=OUTPUT + 'model_cache/',
    device='cuda:0'
)
print("Model loaded ✓")

# ── DataLoader ──
train_dataloader_mono = DataLoader(
    train_examples_mono,
    shuffle=True,
    batch_size=32
)
train_loss_mono = MultipleNegativesRankingLoss(model_mono)

print("Steps: " + str(len(train_dataloader_mono)))
print("Starting monolingual training...")

# ── Train ──
model_mono.fit(
    train_objectives=[(train_dataloader_mono, train_loss_mono)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

# ── Verify ──
print("\nVerifying...")
test_emb = model_mono.encode(
    ["flood evacuation emergency"],
    normalize_embeddings=True,
    convert_to_numpy=True
)

if np.isnan(test_emb).any():
    print("ERROR: NaN — not saving")
else:
    model_mono.save(OUTPUT + 'mono_model_mpnet/')
    print("Monolingual model saved ✓ — no NaN")

In [ ]:
!pip install faiss-cpu sentence-transformers -q

import numpy as np
import faiss
import torch
import gc
import pandas as pd
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

# ── Load fact-checks ──
train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
all_fc   = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc   = all_fc.drop_duplicates(subset=['fact_check_id']).reset_index(drop=True)
all_fc['fc_text_eng']  = all_fc['fc_text_eng'].fillna('unknown')
all_fc['fc_text_orig'] = all_fc['fc_text_orig'].fillna('unknown')
print("Fact-checks: " + str(len(all_fc)))

# ── Encode with crosslingual model ──
print("\nLoading crosslingual model...")
gc.collect()
torch.cuda.empty_cache()

model_cross = SentenceTransformer(
    OUTPUT + 'cross_model_mpnet/',
    device='cuda:0'
)

print("Encoding crosslingual (english)...")
cross_embs = model_cross.encode(
    all_fc['fc_text_eng'].tolist(),
    batch_size=512,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print("NaN check: " + str(np.isnan(cross_embs).any()))
np.save(OUTPUT + 'cross_fc_embeddings_mpnet.npy', cross_embs)
print("Saved ✓ shape: " + str(cross_embs.shape))

del model_cross, cross_embs
gc.collect()
torch.cuda.empty_cache()

# ── Encode with monolingual model ──
print("\nLoading monolingual model...")
model_mono = SentenceTransformer(
    OUTPUT + 'mono_model_mpnet/',
    device='cuda:0'
)

print("Encoding monolingual (original)...")
mono_embs = model_mono.encode(
    all_fc['fc_text_orig'].tolist(),
    batch_size=512,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print("NaN check: " + str(np.isnan(mono_embs).any()))
np.save(OUTPUT + 'mono_fc_embeddings_mpnet.npy', mono_embs)
print("Saved ✓ shape: " + str(mono_embs.shape))

fc_ids = all_fc['fact_check_id'].tolist()
np.save(OUTPUT + 'fc_ids.npy', np.array(fc_ids))

del model_mono, mono_embs
gc.collect()
torch.cuda.empty_cache()

# ── Build FAISS indexes ──
print("\nBuilding FAISS indexes...")
cross_embs = np.load(OUTPUT + 'cross_fc_embeddings_mpnet.npy').astype('float32')
cross_index = faiss.IndexFlatIP(cross_embs.shape[1])
cross_index.add(cross_embs)
faiss.write_index(cross_index, OUTPUT + 'cross_faiss_mpnet.index')
print("Cross FAISS: " + str(cross_index.ntotal) + " vectors ✓")
del cross_embs, cross_index

mono_embs = np.load(OUTPUT + 'mono_fc_embeddings_mpnet.npy').astype('float32')
mono_index = faiss.IndexFlatIP(mono_embs.shape[1])
mono_index.add(mono_embs)
faiss.write_index(mono_index, OUTPUT + 'mono_faiss_mpnet.index')
print("Mono FAISS : " + str(mono_index.ntotal) + " vectors ✓")
del mono_embs, mono_index

print("\n" + "="*40)
print("All done ✓")
print("Ready for evaluation")
print("="*40)

In [ ]:
import json
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

OUTPUT = '/kaggle/working/'

# ── Load models and indexes ──
print("Loading...")
model_cross = SentenceTransformer(OUTPUT + 'cross_model_mpnet/', device='cuda:0')
model_mono  = SentenceTransformer(OUTPUT + 'mono_model_mpnet/',  device='cuda:0')

fc_ids      = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)
fc_ids_list = [int(x) for x in fc_ids]

cross_index = faiss.read_index(OUTPUT + 'cross_faiss_mpnet.index')
mono_index  = faiss.read_index(OUTPUT + 'mono_faiss_mpnet.index')

train_posts = pd.read_csv(OUTPUT + 'train_posts_parsed.csv')
test_posts  = pd.read_csv(OUTPUT + 'test_posts_parsed.csv')
all_posts   = pd.concat([train_posts, test_posts], ignore_index=True)

dev_pairs_cross  = pd.read_csv(OUTPUT + 'dev_pairs_cross.csv')
dev_pairs_mono   = pd.read_csv(OUTPUT + 'dev_pairs_mono.csv')
test_pairs_cross = pd.read_csv(OUTPUT + 'test_pairs_cross.csv')
test_pairs_mono  = pd.read_csv(OUTPUT + 'test_pairs_mono.csv')

with open(OUTPUT + 'dev_cross_reference.json')  as f: dev_cross_ref  = json.load(f)
with open(OUTPUT + 'dev_mono_reference.json')   as f: dev_mono_ref   = json.load(f)
with open(OUTPUT + 'test_cross_reference.json') as f: test_cross_ref = json.load(f)
with open(OUTPUT + 'test_mono_reference.json')  as f: test_mono_ref  = json.load(f)

print("All loaded ✓")

# ── Retrieval functions ──
def retrieve_cross(query, top_k=10):
    q_vec = model_cross.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True).astype('float32')
    _, I = cross_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in I[0]]

def retrieve_mono(query, top_k=10):
    q_vec = model_mono.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True).astype('float32')
    _, I = mono_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in I[0]]

def success_at_k(results, reference, k=10):
    scores = []
    for post_id, retrieved in results.items():
        if post_id not in reference: continue
        relevant = set(reference[post_id])
        hit = 1 if len(relevant & set(retrieved[:k])) > 0 else 0
        scores.append(hit)
    return round(sum(scores)/len(scores), 4) if scores else 0

# ── DEV evaluation ──
print("\nRunning DEV crosslingual...")
dev_cross_results = {}
for pid in tqdm(dev_pairs_cross['post_id'].unique()):
    row = all_posts[all_posts['post_id']==pid]
    if row.empty: continue
    q = str(row.iloc[0]['post_text_eng']).strip()
    dev_cross_results[str(pid)] = retrieve_cross(q)

print("Running DEV monolingual...")
dev_mono_results = {}
for pid in tqdm(dev_pairs_mono['post_id'].unique()):
    row = all_posts[all_posts['post_id']==pid]
    if row.empty: continue
    q = str(row.iloc[0]['post_text_orig']).strip()
    dev_mono_results[str(pid)] = retrieve_mono(q)

# ── TEST evaluation ──
print("\nRunning TEST crosslingual...")
test_cross_results = {}
for pid in tqdm(test_pairs_cross['post_id'].unique()):
    row = test_posts[test_posts['post_id']==pid]
    if row.empty: continue
    q = str(row.iloc[0]['post_text_eng']).strip()
    test_cross_results[str(pid)] = retrieve_cross(q)

print("Running TEST monolingual...")
test_mono_results = {}
for pid in tqdm(test_pairs_mono['post_id'].unique()):
    row = test_posts[test_posts['post_id']==pid]
    if row.empty: continue
    q = str(row.iloc[0]['post_text_orig']).strip()
    test_mono_results[str(pid)] = retrieve_mono(q)

# ── Results ──
dev_cross_s10  = success_at_k(dev_cross_results,  dev_cross_ref)
dev_mono_s10   = success_at_k(dev_mono_results,   dev_mono_ref)
test_cross_s10 = success_at_k(test_cross_results, test_cross_ref)
test_mono_s10  = success_at_k(test_mono_results,  test_mono_ref)

print("\n" + "="*50)
print("FINAL EVALUATION RESULTS")
print("="*50)
print("                   DEV      TEST")
print("Crosslingual S@10: " + str(dev_cross_s10) + "    " + str(test_cross_s10))
print("Monolingual  S@10: " + str(dev_mono_s10)  + "    " + str(test_mono_s10))
print("="*50)
print("\nBASELINE COMPARISON (TEST):")
print("BM25              Cross:0.45  Mono:0.65")
print("GTR-T5-Large      Cross:0.58  Mono:0.76")
print("Multilingual-E5   Cross:0.66  Mono:0.83")
print("YOUR SYSTEM       Cross:" + str(test_cross_s10) + "  Mono:" + str(test_mono_s10))

In [ ]:
import json, faiss, numpy as np
import pandas as pd, torch
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

# ── Load models ──
model_cross = SentenceTransformer(OUTPUT + 'cross_model_mpnet/', device='cuda:0')
model_mono  = SentenceTransformer(OUTPUT + 'mono_model_mpnet/',  device='cuda:0')

fc_ids      = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)
fc_ids_list = [int(x) for x in fc_ids]

cross_index = faiss.read_index(OUTPUT + 'cross_faiss_mpnet.index')
mono_index  = faiss.read_index(OUTPUT + 'mono_faiss_mpnet.index')

# ── Load fact-checks for display ──
train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
all_fc   = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc   = all_fc.drop_duplicates(subset=['fact_check_id']).reset_index(drop=True)
fc_lookup = dict(zip(all_fc['fact_check_id'], all_fc['fc_text_eng']))

print("System ready ✓")

# ── Retrieval functions ──
def search_crosslingual(query_english, top_k=5):
    q_vec = model_cross.encode(
        [query_english],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    _, I = cross_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in I[0]]

def search_monolingual(query_original, top_k=5):
    q_vec = model_mono.encode(
        [query_original],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    _, I = mono_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in I[0]]

def show_results(query, mode='cross'):
    print("\n" + "="*60)
    print("MODE    : " + mode.upper())
    print("QUERY   : " + query)
    print("="*60)

    if mode == 'cross':
        results = search_crosslingual(query)
    else:
        results = search_monolingual(query)

    for rank, fc_id in enumerate(results, 1):
        text = fc_lookup.get(fc_id, 'N/A')
        print("\n#" + str(rank) + " [ID: " + str(fc_id) + "]")
        print("   " + str(text)[:120] + "...")

# ── Test queries ──

# English crosslingual
show_results(
    "floods in Germany killed 600 babies",
    mode='cross'
)

# Urdu crosslingual
show_results(
    "جرمنی میں سیلاب سے 600 بچے ہلاک",
    mode='cross'
)

# Arabic crosslingual
show_results(
    "الفيضانات في ألمانيا قتلت 600 طفل",
    mode='cross'
)

# Spanish monolingual
show_results(
    "Las inundaciones en Alemania mataron 600 bebés",
    mode='mono'
)

# French monolingual
show_results(
    "Les inondations en Allemagne ont tué 600 bébés",
    mode='mono'
)

In [ ]:
# ── Different topic test queries ──

# COVID vaccine crosslingual (English)
show_results(
    "COVID vaccine causes infertility and DNA changes",
    mode='cross'
)

# COVID vaccine crosslingual (Arabic)
show_results(
    "لقاح كوفيد يسبب العقم ويغير الحمض النووي",
    mode='cross'
)

# Earthquake crosslingual (English)
show_results(
    "earthquake in Turkey killed thousands buildings collapsed",
    mode='cross'
)

# 5G towers spreading coronavirus crosslingual
show_results(
    "5G towers are spreading coronavirus disease",
    mode='cross'
)

# Ukraine war misinformation crosslingual
show_results(
    "Ukraine war footage is actually from video game",
    mode='cross'
)

# Spanish monolingual
show_results(
    "La vacuna del COVID causa infertilidad",
    mode='mono'
)

# French monolingual
show_results(
    "Les tours 5G propagent le coronavirus",
    mode='mono'
)

# Arabic monolingual
show_results(
    "برج 5G ينشر فيروس كورونا",
    mode='mono'
)

# Portuguese monolingual
show_results(
    "A vacina COVID causa infertilidade",
    mode='mono'
)

# German monolingual
show_results(
    "5G Türme verbreiten das Coronavirus",
    mode='mono'
)

In [ ]:
import os
import numpy as np
import faiss
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

# ── Check all files ──
print("FILES IN /kaggle/working/:")
print("="*50)
for item in sorted(os.listdir(OUTPUT)):
    path = OUTPUT + item
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024*1024)
        print(f"  FILE  {item:<45} {size:.1f} MB")
    elif os.path.isdir(path):
        dir_size = sum(
            os.path.getsize(os.path.join(r, f))
            for r, d, files in os.walk(path)
            for f in files
        ) / (1024*1024)
        print(f"  DIR   {item:<45} {dir_size:.1f} MB")
print("="*50)

# ── Load everything ──
print("\nLoading system...")
!pip install faiss-cpu sentence-transformers -q

model_cross = SentenceTransformer(OUTPUT + 'cross_model_mpnet/', device='cuda:0')
model_mono  = SentenceTransformer(OUTPUT + 'mono_model_mpnet/',  device='cuda:0')

fc_ids      = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)
fc_ids_list = [int(x) for x in fc_ids]

cross_index = faiss.read_index(OUTPUT + 'cross_faiss_mpnet.index')
mono_index  = faiss.read_index(OUTPUT + 'mono_faiss_mpnet.index')

train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
all_fc   = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc   = all_fc.drop_duplicates(subset=['fact_check_id']).reset_index(drop=True)
fc_lookup = dict(zip(all_fc['fact_check_id'], all_fc['fc_text_eng']))

print("System loaded ✓")
print(f"Fact-checks indexed: {cross_index.ntotal:,}")

# ── Retrieval functions ──
def search(query, mode='cross', top_k=5):
    if mode == 'cross':
        q_vec = model_cross.encode(
            [query], normalize_embeddings=True,
            convert_to_numpy=True).astype('float32')
        _, I = cross_index.search(q_vec, top_k)
    else:
        q_vec = model_mono.encode(
            [query], normalize_embeddings=True,
            convert_to_numpy=True).astype('float32')
        _, I = mono_index.search(q_vec, top_k)

    print("\n" + "="*60)
    print("MODE  : " + mode.upper())
    print("QUERY : " + query)
    print("="*60)
    for rank, idx in enumerate(I[0], 1):
        fc_id = fc_ids_list[idx]
        text  = fc_lookup.get(fc_id, 'N/A')
        print(f"\n#{rank} [ID: {fc_id}]")
        print(f"   {text}")

# ── Test queries ──
search("5G towers spreading coronavirus", mode='cross')
search("COVID vaccine causes infertility", mode='cross')
search("floods in Germany killed babies", mode='cross')
search("La vacuna COVID causa infertilidad", mode='mono')
search("Les tours 5G propagent le coronavirus", mode='mono')

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'faiss-cpu', 'sentence-transformers', '-q'])

import os
import numpy as np
import faiss
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

OUTPUT = '/kaggle/working/'

# ── Check all files ──
print("FILES IN /kaggle/working/:")
print("="*50)
for item in sorted(os.listdir(OUTPUT)):
    path = OUTPUT + item
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024*1024)
        print("  FILE  " + item.ljust(45) + str(round(size,1)) + " MB")
    elif os.path.isdir(path):
        dir_size = sum(
            os.path.getsize(os.path.join(r, f))
            for r, d, files in os.walk(path)
            for f in files
        ) / (1024*1024)
        print("  DIR   " + item.ljust(45) + str(round(dir_size,1)) + " MB")
print("="*50)

# ── Load everything ──
print("\nLoading system...")
model_cross = SentenceTransformer(OUTPUT + 'cross_model_mpnet/', device='cuda:0')
model_mono  = SentenceTransformer(OUTPUT + 'mono_model_mpnet/',  device='cuda:0')

fc_ids      = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)
fc_ids_list = [int(x) for x in fc_ids]

cross_index = faiss.read_index(OUTPUT + 'cross_faiss_mpnet.index')
mono_index  = faiss.read_index(OUTPUT + 'mono_faiss_mpnet.index')

train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
all_fc   = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc   = all_fc.drop_duplicates(subset=['fact_check_id']).reset_index(drop=True)
fc_lookup = dict(zip(all_fc['fact_check_id'], all_fc['fc_text_eng']))

print("System loaded ✓")
print("Fact-checks indexed: " + str(cross_index.ntotal))

# ── Retrieval functions ──
def search(query, mode='cross', top_k=5):
    if mode == 'cross':
        q_vec = model_cross.encode(
            [query], normalize_embeddings=True,
            convert_to_numpy=True).astype('float32')
        _, I = cross_index.search(q_vec, top_k)
    else:
        q_vec = model_mono.encode(
            [query], normalize_embeddings=True,
            convert_to_numpy=True).astype('float32')
        _, I = mono_index.search(q_vec, top_k)

    print("\n" + "="*60)
    print("MODE  : " + mode.upper())
    print("QUERY : " + query)
    print("="*60)
    for rank, idx in enumerate(I[0], 1):
        fc_id = fc_ids_list[idx]
        text  = fc_lookup.get(fc_id, 'N/A')
        print("\n#" + str(rank) + " [ID: " + str(fc_id) + "]")
        print("   " + str(text))

# ── Test queries ──
search("5G towers spreading coronavirus", mode='cross')
search("COVID vaccine causes infertility", mode='cross')
search("floods in Germany killed babies", mode='cross')
search("La vacuna COVID causa infertilidad", mode='mono')
search("Les tours 5G propagent le coronavirus", mode='mono')

In [5]:
import os
import subprocess
subprocess.run(['pip', 'install', 'huggingface_hub', 'sentence-transformers', '-q'])

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login
from sentence_transformers import SentenceTransformer

# ── Auth ──
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
print("HuggingFace logged in ✓")

OUTPUT = '/kaggle/working/'
HF_USERNAME = "Sayyam-1"

# ── Push crosslingual model ──
print("\nUploading crosslingual model...")
model_cross = SentenceTransformer(OUTPUT + 'cross_model_mpnet/')
model_cross.push_to_hub(
    f"{HF_USERNAME}/crislens-cross-mpnet",
    token=HF_TOKEN
)
print("cross_model_mpnet ✓")
del model_cross

# ── Push monolingual model ──
print("\nUploading monolingual model...")
model_mono = SentenceTransformer(OUTPUT + 'mono_model_mpnet/')
model_mono.push_to_hub(
    f"{HF_USERNAME}/crislens-mono-mpnet",
    token=HF_TOKEN
)
print("mono_model_mpnet ✓")
del model_mono

# ── Create dataset repo ──
DATASET_REPO = f"{HF_USERNAME}/crislens-artifacts"
api.create_repo(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    exist_ok=True,
    private=False,
    token=HF_TOKEN
)
print("\nDataset repo created ✓")

# ── Upload artifact files ──
artifacts = [
    "cross_fc_embeddings_mpnet.npy",
    "mono_fc_embeddings_mpnet.npy",
    "cross_faiss_mpnet.index",
    "mono_faiss_mpnet.index",
    "fc_ids.npy",
    "train_posts_parsed.csv",
    "train_fc_parsed.csv",
    "test_posts_parsed.csv",
    "test_fc_parsed.csv",
    "train_pairs.csv",
    "dev_pairs_cross.csv",
    "dev_pairs_mono.csv",
    "test_pairs_cross.csv",
    "test_pairs_mono.csv",
    "dev_cross_reference.json",
    "dev_mono_reference.json",
    "test_cross_reference.json",
    "test_mono_reference.json",
]

print(f"\nUploading {len(artifacts)} files...")

for filename in artifacts:
    path = OUTPUT + filename
    if not os.path.exists(path):
        print(f"  SKIP: {filename}")
        continue
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {filename} ({size_mb:.1f} MB)...", end=" ")
    api.upload_file(
        path_or_fileobj=path,
        path_in_repo=filename,
        repo_id=DATASET_REPO,
        repo_type="dataset",
        token=HF_TOKEN
    )
    print("✓")

print("\n" + "="*50)
print("ALL UPLOADED ✓")
print(f"Models   : huggingface.co/{HF_USERNAME}/crislens-cross-mpnet")
print(f"Artifacts: huggingface.co/datasets/{HF_USERNAME}/crislens-artifacts")
print("="*50)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace logged in ✓

Uploading crosslingual model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

✓
  test_fc_parsed.csv (388.5 MB)... 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓
  train_pairs.csv (0.3 MB)... ✓
  dev_pairs_cross.csv (0.0 MB)... ✓
  dev_pairs_mono.csv (0.0 MB)... ✓
  test_pairs_cross.csv (0.1 MB)... ✓
  test_pairs_mono.csv (0.1 MB)... ✓
  dev_cross_reference.json (0.0 MB)... ✓
  dev_mono_reference.json (0.0 MB)... ✓
  test_cross_reference.json (0.1 MB)... ✓
  test_mono_reference.json (0.1 MB)... ✓

ALL UPLOADED ✓
Models   : huggingface.co/Sayyam-1/crislens-cross-mpnet
Artifacts: huggingface.co/datasets/Sayyam-1/crislens-artifacts
